# Attention Visualisiert

Dieses Notebook baut einen winzigen Attention-Layer und lässt
dich live dabei zusehen. Du wirst sehen, welche Wörter welchen
anderen Wörtern Aufmerksamkeit schenken und wie die Causal Mask
die Zukunft blockiert. Jede Zahl ist real. Jede Heatmap zeigt
tatsächliche Attention-Gewichte.

Wir schicken den Satz *The cat sat on the mat* durch einen
einzelnen Attention Head und stellen die Gewichtsmatrix als
Heatmap dar. Danach fügen wir weitere Heads hinzu und sehen,
wie sich jeder von ihnen spezialisiert.

Kein Training nötig. Die Gewichte sind zufällig. Aber die Muster
sind real. Die Causal Mask blockiert immer die Zukunft. Der
Softmax sorgt immer dafür, dass jede Zeile in Summe eins ergibt.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

## Ein winziges Attention-Modul bauen

Dies ist genau derselbe Attention-Code wie im Guide, nur
angepasst, damit er die Attention-Gewichte zur Visualisierung
zurückgibt.

In [ ]:
class SingleHeadAttention(nn.Module):
    def __init__(self, d_model, head_dim):
        super().__init__()
        self.head_dim = head_dim
        self.w_q = nn.Linear(d_model, head_dim, bias=False)
        self.w_k = nn.Linear(d_model, head_dim, bias=False)
        self.w_v = nn.Linear(d_model, head_dim, bias=False)

    def forward(self, x, causal=True, return_weights=False):
        batch, seq_len, d_model = x.shape
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if causal:
            mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
            scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)
        output = weights @ v

        if return_weights:
            return output, weights
        return output


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, causal=True, return_weights=False):
        batch, seq_len, d_model = x.shape
        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if causal:
            mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
            mask = mask.view(1, 1, seq_len, seq_len)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)
        output = weights @ v
        output = output.transpose(1, 2).contiguous()
        output = output.reshape(batch, seq_len, d_model)
        output = self.out_proj(output)

        if return_weights:
            return output, weights
        return output

## Ein winziges Embedding für unseren Satz erstellen

Wir verwenden kleine zufällige Vektoren anstelle einer echten
Embedding-Tabelle. Die Attention-Muster funktionieren trotzdem
genauso.

In [ ]:
torch.manual_seed(42)
words = ["The", "cat", "sat", "on", "the", "mat"]
seq_len = len(words)

d_model = 768
x = torch.randn(1, seq_len, d_model)

print(f"Sequence length: {seq_len}")
print(f"Input shape: {x.shape}")
print(f"Words: {words}")

## Heatmap für Single-Head-Attention

Ein Attention Head erzeugt eine Gewichtsmatrix. Jede Zeile ist
ein Token in der Rolle der Query. Jede Spalte ist ein Token in
der Rolle des Key. Die Farbe zeigt, wie viel Aufmerksamkeit
Token i dem Token j schenkt.

In [ ]:
attn_single = SingleHeadAttention(d_model, head_dim=64)
_, weights = attn_single(x, causal=True, return_weights=True)

w = weights[0].detach().numpy()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(w, cmap='YlOrRd', vmin=0, vmax=1)

ax.set_xticks(range(seq_len))
ax.set_xticklabels(words, rotation=45, ha='right')
ax.set_yticks(range(seq_len))
ax.set_yticklabels(words)
ax.set_xlabel('Key (what I offer)')
ax.set_ylabel('Query (what I am looking for)')
ax.set_title('Single Head Attention Weights')

for i in range(seq_len):
    for j in range(seq_len):
        color = 'white' if w[i, j] > 0.5 else 'black'
        ax.text(j, i, f'{w[i, j]:.2f}', ha='center', va='center', color=color, fontsize=9)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

print("Upper right is all zeros.")
print("The causal mask blocks tokens from seeing the future.")
print(f"Token 0 sees only itself: {w[0, 0]:.2f}")
print(f"Token 5 sees all six tokens: max={w[5].max():.2f}")

## Was ohne die Causal Mask passiert

Schalten wir die Maske aus, kann jedes Token jedes andere Token
sehen. Das ist es, was ein Encoder wie BERT verwendet. Aber ein
GPT darf das während des Trainings nicht tun, sonst würde es
schummeln.

In [ ]:
_, weights_no_mask = attn_single(x, causal=False, return_weights=True)

w = weights_no_mask[0].detach().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

_, weights_causal = attn_single(x, causal=True, return_weights=True)
w_causal = weights_causal[0].detach().numpy()

ax1.imshow(w_causal, cmap='YlOrRd', vmin=0, vmax=1)
ax1.set_xticks(range(seq_len))
ax1.set_xticklabels(words, rotation=45, ha='right')
ax1.set_yticks(range(seq_len))
ax1.set_yticklabels(words)
ax1.set_title('With causal mask (GPT)')

ax2.imshow(w, cmap='YlOrRd', vmin=0, vmax=1)
ax2.set_xticks(range(seq_len))
ax2.set_xticklabels(words, rotation=45, ha='right')
ax2.set_yticks(range(seq_len))
ax2.set_yticklabels(words)
ax2.set_title('Without causal mask (no cheating prevention)')

plt.tight_layout()
plt.show()

## Multi-Head-Attention: zwölf Heads

Jeder Head lernt ein anderes Muster. Selbst bei zufälligen
Gewichten sieht man, dass jeder Head eine unterschiedliche
Gewichtsmatrix erzeugt. In einem trainierten Modell würden diese
Unterschiede tatsächlichen sprachlichen Mustern entsprechen, wie
etwa der Kongruenz zwischen Subjekt und Verb.

In [ ]:
attn_multi = MultiHeadAttention(d_model=768, num_heads=12)
_, weights = attn_multi(x, causal=True, return_weights=True)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for h in range(12):
    w = weights[0, h].detach().numpy()
    ax = axes[h]
    im = ax.imshow(w, cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xticks(range(seq_len))
    ax.set_xticklabels(words, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(seq_len))
    ax.set_yticklabels(words, fontsize=7)
    ax.set_title(f'Head {h + 1}', fontsize=10)

plt.suptitle('All 12 Attention Heads in One Transformer Layer', fontsize=14)
plt.tight_layout()
plt.show()

print("Each head sees the same input but produces different attention patterns.")
print("The causal mask is always present. Every head has the upper right blocked.")

## Probiere deinen eigenen Satz aus

Ändere die Wörter unten und führe die Zellen erneut aus, um zu
sehen, wie sich die Attention bei unterschiedlichen Eingaben
verhält. Die Muster ändern sich, weil das zufällige Embedding
jedes Wortes anders ist. Mit echten, trainierten Embeddings
würdest du bedeutungsvolle sprachliche Muster sehen.

In [ ]:
your_words = ["The", "dog", "ran", "through", "the", "park"]

x2 = torch.randn(1, len(your_words), 768)
_, weights = attn_multi(x2, causal=True, return_weights=True)

fig, axes = plt.subplots(2, 6, figsize=(20, 7))
axes = axes.flatten()

for h in range(12):
    w = weights[0, h].detach().numpy()
    ax = axes[h]
    ax.imshow(w, cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_xticks(range(len(your_words)))
    ax.set_xticklabels(your_words, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(len(your_words)))
    ax.set_yticklabels(your_words, fontsize=7)
    ax.set_title(f'Head {h + 1}', fontsize=9)

plt.suptitle(f'Attention patterns for: {" ".join(your_words)}', fontsize=14)
plt.tight_layout()
plt.show()

## Was dir diese Heatmaps zeigen

Das obere rechte Dreieck ist immer leer. Die Causal Mask
verhindert, dass ein Token Wörter sieht, die nach ihm kommen.
Das ist es, was aus GPT ein Sprachmodell macht und keinen
Encoder.

Die Diagonale ist oft hell. Jedes Token schenkt sich selbst
Aufmerksamkeit. Self-Attention ist wichtig, weil kein anderes
Token exakt dieselben Informationen besitzt, die du über dich
selbst hast.

Die erste Zeile ist immer vollständig auf Spalte null ausgeprägt.
Token null hat nur sich selbst, worauf es blicken kann. Es muss
seine gesamte Aufmerksamkeit dorthin richten.

In einem trainierten Modell wären die Attention-Muster
bedeutungsvoll. Head drei könnte Subjekte mit ihren Verben
verbinden. Head sieben könnte Pronomen mit den Substantiven
verbinden, auf die sie sich beziehen. Bei zufälligen Gewichten
sind die Muster zufällig, aber die Struktur ist klar erkennbar.
Das Modell hat die Kapazität, diese Muster zu lernen. Es braucht
nur Daten und Training.